# 12. 프론트엔드 스트리밍

## 학습 목표

LLM 응답을 실시간으로 스트리밍하여 사용자에게 전달하는 방법을 알아봅니다.

이 노트북에서 다루는 내용:
- LangChain SDK의 스트리밍 기초(`.stream()`, `.astream_events()`)를 이해한다
- `useStream` React 훅의 구조와 사용법을 안다
- `StreamEvent` 프로토콜을 이해한다
- Python SDK로 실시간 스트리밍을 소비하는 방법을 익힌다
- 에이전트 상태 실시간 표시 패턴을 안다

## 12.1 환경 설정

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("환경 준비 완료.")

환경 준비 완료.


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

Langfuse tracing ON — https://lf.ddok.ai


## 12.2 Python SDK 스트리밍 기초

`.stream()` 메서드는 모델 응답을 토큰 단위로 실시간 전달합니다. 전체 응답이 완성되기 전에 부분 결과를 볼 수 있습니다.

In [3]:
# .stream()으로 토큰 단위 스트리밍
print("스트리밍 응답:")
for chunk in model.stream(
    "서울의 유명한 관광지 3곳을 알려주세요.",
    config=lf_config,
):
    print(chunk.content, end="", flush=True)
print()  # 줄바꿈

스트리밍 응답:


물

론

입니다

!

...

 방문

해

보

세요

!

## 12.3 astream_events()

`.astream_events()`는 비동기 방식으로 **모든 내부 이벤트**를 스트리밍합니다. 모델 호출, 도구 실행, 체인 단계 등을 세밀하게 추적할 수 있습니다.

### 주요 이벤트 타입

| 이벤트 | 설명 |
|--------|------|
| `on_chat_model_stream` | 모델 토큰 스트리밍 |
| `on_chat_model_start` | 모델 호출 시작 |
| `on_chat_model_end` | 모델 호출 완료 |
| `on_tool_start` | 도구 실행 시작 |
| `on_tool_end` | 도구 실행 완료 |

In [4]:
import asyncio

async def stream_events_demo():
    """astream_events()로 이벤트 스트리밍 예시"""
    print("이벤트 스트리밍:")
    print("-" * 40)
    async for event in model.astream_events(
        "파이썬의 장점 2가지",
        version="v2",
    ):
        kind = event["event"]
        if kind == "on_chat_model_stream":
            content = event["data"]["chunk"].content
            if content:
                print(content, end="", flush=True)
        elif kind == "on_chat_model_start":
            print(f"[모델 호출 시작]")
        elif kind == "on_chat_model_end":
            print(f"\n[모델 호출 완료]")

await stream_events_demo()

이벤트 스트리밍:
----------------------------------------
[모델 호출 시작]


파

이

썬

의

...

율

적

입니다

.


[모델 호출 완료]


## 12.4 useStream React 훅

`useStream`은 LangGraph SDK에서 제공하는 React 훅으로, LangGraph 서버와의 스트리밍 통신을 간편하게 처리합니다.

### 기본 사용법

```tsx
import { useStream } from "@langchain/langgraph-sdk/react";

function Chat() {
  const stream = useStream({
    assistantId: "agent",
    apiUrl: "http://localhost:2024",
  });

  const handleSubmit = (message: string) => {
    stream.submit({
      messages: [{ content: message, type: "human" }],
    });
  };

  return (
    <div>
      {stream.messages.map((message, idx) => (
        <div key={message.id ?? idx}>
          {message.type}: {message.content}
        </div>
      ))}
      {stream.isLoading && <div>Loading...</div>}
      {stream.error && <div>Error: {stream.error.message}</div>}
    </div>
  );
}
```

### 주요 반환값

| 속성 | 타입 | 설명 |
|------|------|------|
| `messages` | `Message[]` | 현재 스레드의 전체 메시지 |
| `isLoading` | `boolean` | 스트림 진행 여부 |
| `error` | `Error \| null` | 에러 객체 |
| `interrupt` | `Interrupt` | 중단 요청 (HITL) |
| `submit()` | `function` | 메시지 전송 |
| `stop()` | `function` | 스트림 중단 |

## 12.5 useStream 설정 옵션

| 파라미터 | 필수 | 기본값 | 설명 |
|----------|------|--------|------|
| `assistantId` | O | — | 에이전트 식별자 (배포 대시보드에서 확인) |
| `apiUrl` | — | `localhost:2024` | 에이전트 서버 URL |
| `apiKey` | — | — | 배포된 에이전트 인증 토큰 |
| `threadId` | — | — | 기존 대화 스레드에 연결 |
| `onThreadId` | — | — | 스레드 생성 시 콜백 |
| `reconnectOnMount` | — | `false` | 컴포넌트 마운트 시 진행 중 스트림 재연결 |
| `onCustomEvent` | — | — | 커스텀 이벤트 핸들러 |
| `onUpdateEvent` | — | — | 상태 업데이트 핸들러 |
| `onMetadataEvent` | — | — | 메타데이터 이벤트 핸들러 |
| `messagesKey` | — | `"messages"` | 메시지를 담는 상태 키 |
| `throttle` | — | `true` | 상태 업데이트 배치 처리 |
| `initialValues` | — | — | 캐시된 초기 상태 |

## 12.6 스레드 관리와 재연결

### 스레드 ID 관리

`threadId`를 관리하면 대화를 지속하거나 이전 대화를 불러올 수 있습니다.

```tsx
const [threadId, setThreadId] = useState<string | null>(null);

const stream = useStream({
  apiUrl: "http://localhost:2024",
  assistantId: "agent",
  threadId,
  onThreadId: setThreadId,
});

// threadId를 URL 파라미터나 localStorage에 저장하여 지속성 확보
```

### 페이지 새로고침 후 재연결

`reconnectOnMount`를 활성화하면 페이지 새로고침 후에도 진행 중이던 스트림에 자동 재연결됩니다.

```tsx
const stream = useStream({
  apiUrl: "http://localhost:2024",
  assistantId: "agent",
  reconnectOnMount: true, // sessionStorage 사용
});

// 커스텀 스토리지 사용
const stream = useStream({
  reconnectOnMount: () => window.localStorage,
});
```

## 12.7 브랜칭과 메시지 편집

브랜칭을 쓰면 대화 히스토리의 특정 지점에서 **대체 경로**를 만들 수 있습니다. 메시지를 편집하거나 AI 응답을 재생성할 때 유용합니다.

```tsx
{stream.messages.map((message) => {
  const meta = stream.getMessagesMetadata(message);
  const parentCheckpoint = meta?.firstSeenState?.parent_checkpoint;

  return (
    <div key={message.id}>
      {message.content}

      {/* 사용자 메시지 편집 */}
      {message.type === "human" && (
        <button onClick={() => {
          const newContent = prompt("Edit:", message.content);
          if (newContent) {
            stream.submit(
              { messages: [{ type: "human", content: newContent }] },
              { checkpoint: parentCheckpoint }
            );
          }
        }}>
          Edit
        </button>
      )}

      {/* AI 응답 재생성 */}
      {message.type === "ai" && (
        <button onClick={() =>
          stream.submit(undefined, { checkpoint: parentCheckpoint })
        }>
          Regenerate
        </button>
      )}
    </div>
  );
})}
```

핵심: `checkpoint` 파라미터로 특정 시점의 상태로 돌아가 새로운 분기를 생성합니다.

## 12.8 커스텀 스트리밍 이벤트

에이전트에서 **커스텀 데이터**를 클라이언트로 스트리밍할 수 있습니다. 진행 상황, 중간 결과 등을 실시간으로 전달할 때 유용합니다.

In [5]:
# 커스텀 스트리밍 이벤트 — Python writer 패턴
print("커스텀 스트리밍 이벤트 패턴 (Python 서버 측):")
print("=" * 50)
print("""
from langchain.tools import tool
from langchain.agents.types import ToolRuntime

@tool
async def analyze_data(
    data_source: str, *, config: ToolRuntime
) -> str:
    \"\"\"데이터를 분석합니다.\"\"\"
    if config.writer:
        # 진행 상황을 클라이언트에 스트리밍
        config.writer({
            "type": "progress",
            "message": "데이터 로딩 중...",
            "progress": 25,
        })
        # ... 처리 ...
        config.writer({
            "type": "progress",
            "message": "분석 완료!",
            "progress": 100,
        })
    return '{"result": "분석 완료"}'
""")
print("클라이언트(React) 측: onCustomEvent 콜백으로 수신")
print('  onCustomEvent: (data) => setProgress(data.progress)')

커스텀 스트리밍 이벤트 패턴 (Python 서버 측):

from langchain.tools import tool
from langchain.agents.types import ToolRuntime

@tool
async def analyze_data(
    data_source: str, *, config: ToolRuntime
) -> str:
    """데이터를 분석합니다."""
    if config.writer:
        # 진행 상황을 클라이언트에 스트리밍
        config.writer({
            "type": "progress",
            "message": "데이터 로딩 중...",
            "progress": 25,
        })
        # ... 처리 ...
        config.writer({
            "type": "progress",
            "message": "분석 완료!",
            "progress": 100,
        })
    return '{"result": "분석 완료"}'

클라이언트(React) 측: onCustomEvent 콜백으로 수신
  onCustomEvent: (data) => setProgress(data.progress)


## 12.9 멀티 에이전트 스트리밍

여러 에이전트가 협업하는 환경에서는 각 에이전트의 메시지를 **구분하여 표시**해야 합니다. 메타데이터의 `langgraph_node`로 메시지 출처를 식별합니다.

```tsx
// 노드별 설정
const NODE_CONFIG: Record<string, { label: string; color: string }> = {
  researcher: { label: "Research Agent", color: "blue" },
  writer:     { label: "Writing Agent",  color: "green" },
  reviewer:   { label: "Review Agent",   color: "purple" },
};

// 메시지 렌더링
function AgentMessage({ message, stream }) {
  const metadata = stream.getMessagesMetadata?.(message);
  const nodeName = metadata?.streamMetadata?.langgraph_node;
  const config = NODE_CONFIG[nodeName];

  return (
    <div className={`bg-${config?.color}-950/30 p-4 rounded-lg`}>
      <div className={`text-${config?.color}-400 text-sm font-bold`}>
        {config?.label ?? "Agent"}
      </div>
      <div>{message.content}</div>
    </div>
  );
}
```

### 이벤트 콜백 정리

| 콜백 | 용도 | 스트림 모드 |
|------|------|------------|
| `onUpdateEvent` | 그래프 단계 후 상태 업데이트 | `updates` |
| `onCustomEvent` | 에이전트의 커스텀 이벤트 | `custom` |
| `onMetadataEvent` | 실행 및 스레드 메타데이터 | `metadata` |
| `onError` | 에러 처리 | — |
| `onFinish` | 스트림 완료 | — |

## 12.10 요약

이 노트북에서 배운 내용:

| 주제 | 핵심 내용 |
|------|----------|
| **SDK 스트리밍** | `.stream()`으로 토큰 단위 실시간 응답을 받습니다 |
| **astream_events** | 비동기 이벤트 스트리밍으로 모델/도구 호출을 세밀하게 추적합니다 |
| **useStream** | React 훅으로 LangGraph 서버와 스트리밍 통신을 간편하게 처리합니다 |
| **스레드 관리** | `threadId`와 `reconnectOnMount`로 대화 지속성을 확보합니다 |
| **브랜칭** | `checkpoint` 기반으로 대화의 대체 경로를 생성합니다 |
| **커스텀 이벤트** | `writer` 패턴으로 진행 상황 등 커스텀 데이터를 스트리밍합니다 |
| **멀티에이전트** | `langgraph_node` 메타데이터로 에이전트별 메시지를 구분 표시합니다 |

### 다음 단계
→ **[13. 가드레일](13_guardrails.ipynb)**으로 진행하세요!

---
**참고 문서:**
- [Streaming](../docs/langchain/08-streaming.md)
- [UI (Agent Chat UI & useStream)](../docs/langchain/28-ui.md)